# BIST-100 Strateji Karşılaştırma Motoru
## Klasik TA · GARCH(1,1) · XGBoost — Kim Kazanır?

**Girdi**: `Open`, `High`, `Low`, `Close`, `Volume`, `Endeks_Close` sütunları olan, tarih indeksli temiz bir DataFrame (`df`)

**Çıktı**: 3 stratejinin aynı test döneminde karşılaştırılması + Şampiyon ilan

---
| Adım | İçerik |
|------|--------|
| 1 | Özellik Mühendisliği: RSI, MACD, Hacim Oranı, Göreceli Güç, Label |
| 2 | Strateji A — Klasik TA (RSI + MACD kesişim kuralları) |
| 3 | Strateji B — GARCH(1,1) (oynaklık rejimi + momentum) |
| 4 | Strateji C — XGBoost (ikili sınıflandırma, kronolojik split) |
| 5 | Backtest & Metrikler: Getiri, Sharpe, Max DD, Win Rate |
| 6 | Kıyaslama Tablosu + Şampiyon İlanı |

In [ ]:
import subprocess, sys

def pip_install(pkg, label=None):
    label = label or pkg
    print(f"  {label}...", end=" ")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    print("OK" if r.returncode == 0 else f"HATA\n{r.stderr[-200:]}")

for pkg in [
    "pandas", "numpy", "scipy", "scikit-learn",
    "xgboost",       # XGBoost sınıflandırıcı
    "arch",          # GARCH modeli (arch-py)
    "matplotlib",    # Görselleştirme
    "yfinance",      # Demo verisi için (opsiyonel)
]:
    pip_install(pkg)

print("\nTüm kütüphaneler hazır.")

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from arch import arch_model

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "DejaVu Sans",
})

# ── Global Hiperparametreler ─────────────────────────────────────────────────
TRAIN_RATIO   = 0.80   # %80 eğitim, %20 test (kronolojik)
RSI_PERIOD    = 14
MACD_FAST     = 12
MACD_SLOW     = 26
MACD_SIGNAL   = 9
VOL_WINDOW    = 20     # Hacim ort. penceresi
GARCH_VOL_LOW = 0.35   # Alt volatilite eşiği (percentile)
GARCH_VOL_HIGH= 0.70   # Üst volatilite eşiği (percentile)
MOM_WINDOW    = 5      # GARCH momentum penceresi (gün)

# XGBoost için kullanılacak özellik sütunları
FEATURES = ["RSI", "MACD", "MACD_signal", "MACD_hist", "Vol_ratio", "Rel_strength"]

print("Konfigürasyon yüklendi.")
print(f"  TRAIN_RATIO : {TRAIN_RATIO}")
print(f"  FEATURES    : {FEATURES}")

## Adım 1 — Özellik Mühendisliği & Etiketleme

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Girdi df: Open, High, Low, Close, Volume, Endeks_Close sütunları.
    Çıktı  : Teknik özellikler + hedef değişken (Label) eklenmiş df.

    Özellikler
    ----------
    log_ret      : Günlük logaritmik getiri
    RSI          : RSI(14)
    MACD         : MACD farkı (EMA12 - EMA26)
    MACD_signal  : MACD sinyal çizgisi EMA(9)
    MACD_hist    : MACD histogramı (MACD - signal)
    Vol_ratio    : Hacim / 20G ortalama hacim
    Rel_strength : Close / Endeks_Close (göreli güç)
    Label        : 1 = ertesi gün log getiri > 0, 0 = değil
    """
    df = df.copy()
    df.sort_index(inplace=True)

    # ── Logaritmik Getiri ────────────────────────────────────────────────────
    df["log_ret"] = np.log(df["Close"] / df["Close"].shift(1))

    # ── Hedef Değişken (Label) ───────────────────────────────────────────────
    # Bir sonraki günün log getirisi pozitifse 1, değilse 0
    df["Label"] = (df["log_ret"].shift(-1) > 0).astype(int)

    # ── RSI(14) ──────────────────────────────────────────────────────────────
    delta = df["Close"].diff()
    gain  = delta.where(delta > 0, 0.0).rolling(RSI_PERIOD).mean()
    loss  = (-delta).where(delta < 0, 0.0).rolling(RSI_PERIOD).mean()
    rs    = gain / (loss + 1e-9)
    df["RSI"] = 100 - (100 / (1 + rs))

    # ── MACD (12, 26, 9) ─────────────────────────────────────────────────────
    ema_fast        = df["Close"].ewm(span=MACD_FAST,  adjust=False).mean()
    ema_slow        = df["Close"].ewm(span=MACD_SLOW,  adjust=False).mean()
    df["MACD"]        = ema_fast - ema_slow
    df["MACD_signal"] = df["MACD"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    df["MACD_hist"]   = df["MACD"] - df["MACD_signal"]

    # ── Hacim Değişim Oranı ──────────────────────────────────────────────────
    vol_ma          = df["Volume"].rolling(VOL_WINDOW).mean()
    df["Vol_ratio"] = df["Volume"] / (vol_ma + 1e-9)

    # ── Endeks Göreceli Gücü ─────────────────────────────────────────────────
    df["Rel_strength"] = df["Close"] / (df["Endeks_Close"] + 1e-9)

    # ── NaN Temizliği ────────────────────────────────────────────────────────
    df.dropna(inplace=True)
    df.reset_index(drop=False, inplace=True)
    df.set_index(df.columns[0], inplace=True)   # Tarih indeksini koru

    print(f"  Özellik mühendisliği tamamlandı → {len(df)} satır, {df.shape[1]} sütun")
    return df


# Test
# df_feat = feature_engineering(df)
# df_feat[["Close", "log_ret", "RSI", "MACD", "Vol_ratio", "Rel_strength", "Label"]].tail(3)

## Adım 2 — Strateji Motorları

In [ ]:
# ── Strateji A: Klasik TA (RSI + MACD Kesişimi) ─────────────────────────────
def strategy_a_classic_ta(df: pd.DataFrame) -> pd.Series:
    """
    RSI + MACD kural tabanlı strateji.

    Alım koşulu  : MACD, sinyal çizgisini yukarı keser VE RSI < 70
    Satım koşulu : MACD, sinyal çizgisini aşağı keser  VEYA RSI > 75

    Pozisyon: 1 = Long, 0 = Flat (açığa satış yok)
    """
    # MACD kesişim sinyalleri
    macd_cross_up   = (
        (df["MACD"] > df["MACD_signal"]) &
        (df["MACD"].shift(1) <= df["MACD_signal"].shift(1))
    )
    macd_cross_down = (
        (df["MACD"] < df["MACD_signal"]) &
        (df["MACD"].shift(1) >= df["MACD_signal"].shift(1))
    )

    position   = np.zeros(len(df), dtype=int)
    in_pos     = False

    for i in range(1, len(df)):
        rsi_now = df["RSI"].iloc[i]

        if not in_pos:
            # Alım: MACD yukarı kesişim + RSI aşırı alım bölgesinde değil
            if macd_cross_up.iloc[i] and rsi_now < 70:
                in_pos      = True
                position[i] = 1
        else:
            if macd_cross_down.iloc[i] or rsi_now > 75:
                # Satım: MACD aşağı kesişim veya RSI aşırı alımda
                in_pos      = False
                position[i] = 0
            else:
                position[i] = 1

    return pd.Series(position, index=df.index, name="pos_A")


print("Strateji A (Klasik TA) tanımlandı.")

In [ ]:
# ── Strateji B: GARCH(1,1) Oynaklık Rejimi ──────────────────────────────────
def strategy_b_garch(df: pd.DataFrame, train_end: int) -> pd.Series:
    """
    arch kütüphanesi ile GARCH(1,1) modeli.

    Fikir
    -----
    Koşullu oynaklık (sigma_t) düşükse → piyasa sakin → trendler devam eder.
    Koşullu oynaklık yüksekse  → türbülanslı  → riskten kaçın, flat kal.
    Yön tahmini için MOM_WINDOW günlük momentum kullanılır.

    Kural
    -----
    sigma_t < low_thr  (alt %35) VE momentum > 0  → Long (1)
    sigma_t > high_thr (üst %30)                  → Flat (0)
    Diğer                                          → Momentum yönü
    """
    # GARCH(1,1): yüzde olarak daha stabil (log_ret × 100)
    ret_pct = df["log_ret"] * 100

    # Modeli tüm seri üzerine fit et (sıfırlama maliyetini önle)
    garch = arch_model(ret_pct, vol="Garch", p=1, q=1, dist="normal", mean="Zero")
    res   = garch.fit(disp="off", show_warning=False)
    cond_vol = res.conditional_volatility   # Koşullu standart sapma serisi

    # Eşikler: sadece train setinden hesapla (veri sızıntısını önle)
    train_vol   = cond_vol.iloc[:train_end]
    low_thr     = float(train_vol.quantile(GARCH_VOL_LOW))
    high_thr    = float(train_vol.quantile(GARCH_VOL_HIGH))

    # MOM_WINDOW günlük momentum
    mom = df["Close"].pct_change(MOM_WINDOW)

    position = np.zeros(len(df), dtype=int)

    for i in range(MOM_WINDOW, len(df)):
        v   = cond_vol.iloc[i]
        m   = mom.iloc[i]
        if pd.isna(v) or pd.isna(m):
            continue

        if v > high_thr:
            position[i] = 0                    # Yüksek vol → Flat
        elif v < low_thr and m > 0:
            position[i] = 1                    # Düşük vol + pozitif mom → Long
        else:
            position[i] = 1 if m > 0 else 0   # Orta vol: momentum yönü

    return pd.Series(position, index=df.index, name="pos_B")


print("Strateji B (GARCH) tanımlandı.")

In [ ]:
# ── Strateji C: XGBoost İkili Sınıflandırma ─────────────────────────────────
def strategy_c_xgboost(df: pd.DataFrame) -> tuple[pd.Series, int, float]:
    """
    XGBClassifier ile yarınki kapanış yönü tahmini.

    Kronolojik split: İlk %80 eğitim, son %20 test.
    Veri sızıntısı yoktur: test tahminleri train dışı dönemden gelir.

    Çıktı
    -----
    (position Series, train_end_idx, test_accuracy)
    """
    n         = len(df)
    train_end = int(n * TRAIN_RATIO)

    X = df[FEATURES].values
    y = df["Label"].values

    X_train, X_test = X[:train_end], X[train_end:]
    y_train, y_test = y[:train_end], y[train_end:]

    model = XGBClassifier(
        n_estimators    = 300,
        max_depth       = 4,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        eval_metric     = "logloss",
        random_state    = 42,
        verbosity       = 0,
    )
    model.fit(X_train, y_train)

    y_pred   = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # Pozisyon: sadece test döneminde anlamlı; train dönemi 0
    position              = np.zeros(n, dtype=int)
    position[train_end:]  = y_pred

    pos_series = pd.Series(position, index=df.index, name="pos_C")

    print(f"  XGBoost test doğruluğu: %{accuracy*100:.2f}")
    print(f"  Özellik önemleri:")
    importances = pd.Series(model.feature_importances_, index=FEATURES)
    for feat, imp in importances.sort_values(ascending=False).items():
        bar = "█" * int(imp * 40)
        print(f"    {feat:<15} {bar} {imp:.3f}")

    return pos_series, train_end, accuracy


print("Strateji C (XGBoost) tanımlandı.")

## Adım 3 — Backtest Motoru

In [ ]:
def calc_metrics(position: pd.Series, log_returns: pd.Series,
               test_start_idx: int, strategy_name: str = "") -> dict:
    """
    Vektörel backtest: Sadece test dönemine uygulanır.

    Varsayımlar
    -----------
    - Gün sonu sinyal → ertesi gün açılışta işlem (1 gün gecikme)
    - İşlem maliyeti yok (net performans üst sınırı)
    - Long-only (açığa satış yok)

    Metrikler
    ---------
    Toplam Getiri (%) : Bileşik log getiriden hesaplanan mutlak getiri
    Sharpe Oranı      : Yıllıklaştırılmış (252 iş günü)
    Max Drawdown (%)  : En büyük tepe-dip düşüşü
    Win Rate (%)      : Pozisyonda geçen günlerin kazançlı olanlarının oranı
    """
    # Test dönemi pozisyon ve getiri
    pos_test = position.iloc[test_start_idx:].shift(1).fillna(0)  # 1 gün gecikme
    ret_test = log_returns.iloc[test_start_idx:]

    # Strateji günlük getirileri
    strat_ret = pos_test * ret_test

    # ── Toplam Getiri ────────────────────────────────────────────────────────
    total_return = (np.expm1(strat_ret.sum())) * 100

    # ── Sharpe Oranı ─────────────────────────────────────────────────────────
    daily_std = strat_ret.std()
    sharpe    = (strat_ret.mean() / daily_std * np.sqrt(252)) if daily_std > 1e-9 else 0.0

    # ── Maksimum Drawdown ────────────────────────────────────────────────────
    cum_val      = np.exp(strat_ret.cumsum())
    rolling_peak = cum_val.cummax()
    drawdown     = (cum_val - rolling_peak) / rolling_peak
    max_dd       = float(drawdown.min()) * 100

    # ── Win Rate ─────────────────────────────────────────────────────────────
    active_days = strat_ret[pos_test > 0]
    win_rate    = float((active_days > 0).mean() * 100) if len(active_days) > 0 else 0.0

    # ── Ekstra ───────────────────────────────────────────────────────────────
    n_trades  = int((pos_test.diff().fillna(0) > 0).sum())   # Alım sayısı
    n_days    = int((pos_test > 0).sum())                     # Pozisyonda gün

    return {
        "Toplam Getiri (%)": round(float(total_return), 2),
        "Sharpe Oranı"     : round(float(sharpe), 3),
        "Max Drawdown (%)": round(float(max_dd), 2),
        "Win Rate (%)"     : round(float(win_rate), 2),
        "_n_trades"        : n_trades,
        "_n_days"          : n_days,
        "_strat_ret"       : strat_ret,   # Görselleştirme için (tabloya dahil edilmez)
    }


def cumulative_returns(strat_ret_series: pd.Series) -> pd.Series:
    """Log getirilerden kümülatif değer serisi (1.0'dan başlar)."""
    return np.exp(strat_ret_series.cumsum())


print("Backtest motoru hazır.")

## Adım 4 — Stratejileri Çalıştır & Karşılaştır

In [ ]:
def run_comparison(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """
    3 stratejiyi aynı test döneminde backtest eder.

    1. Özellik mühendisliği
    2. XGBoost kronolojik split → test dönemini belirler
    3. Tüm stratejiler aynı test döneminde çalışır
    4. Metrik tablosu + şampiyon

    Çıktı: (karşılaştırma DataFrame, şampiyon strateji adı)
    """
    SEP = "=" * 68

    print(SEP)
    print("  BIST STRATEJİ KARŞILAŞTIRMA SİSTEMİ")
    print(SEP)

    # ── Özellik Mühendisliği ─────────────────────────────────────────────────
    print("\n[1/5] Özellik mühendisliği...")
    df_f = feature_engineering(df)
    n    = len(df_f)
    t    = int(n * TRAIN_RATIO)

    print(f"  Toplam gün  : {n}")
    print(f"  Train       : {t} gün "
          f"({df_f.index[0].strftime('%Y-%m-%d')} — {df_f.index[t-1].strftime('%Y-%m-%d')})")
    print(f"  Test        : {n-t} gün "
          f"({df_f.index[t].strftime('%Y-%m-%d')} — {df_f.index[-1].strftime('%Y-%m-%d')})")

    results    = {}
    strat_rets = {}

    # ── Strateji C: XGBoost (önce; test indeksini o belirler) ────────────────
    print("\n[2/5] Strateji C: XGBoost eğitiliyor...")
    pos_c, train_end, xgb_acc = strategy_c_xgboost(df_f)
    metrics_c = calc_metrics(pos_c, df_f["log_ret"], train_end, "XGBoost")
    results["C: XGBoost"]    = metrics_c
    strat_rets["C: XGBoost"] = metrics_c.pop("_strat_ret")
    metrics_c.pop("_n_trades", None)
    metrics_c.pop("_n_days", None)

    # ── Strateji A: Klasik TA ────────────────────────────────────────────────
    print("\n[3/5] Strateji A: Klasik TA (RSI + MACD)...")
    pos_a    = strategy_a_classic_ta(df_f)
    metrics_a = calc_metrics(pos_a, df_f["log_ret"], train_end, "Klasik TA")
    results["A: Klasik TA"]   = metrics_a
    strat_rets["A: Klasik TA"] = metrics_a.pop("_strat_ret")
    n_trades_a = metrics_a.pop("_n_trades", 0)
    metrics_a.pop("_n_days", None)
    print(f"  Toplam Getiri: %{metrics_a['Toplam Getiri (%)']:.2f} | "
          f"Win Rate: %{metrics_a['Win Rate (%)']:.1f} | "
          f"İşlem: {n_trades_a}")

    # ── Strateji B: GARCH ────────────────────────────────────────────────────
    print("\n[4/5] Strateji B: GARCH(1,1) fit ediliyor... (30-60 sn sürebilir)")
    try:
        pos_b     = strategy_b_garch(df_f, train_end)
        metrics_b = calc_metrics(pos_b, df_f["log_ret"], train_end, "GARCH")
        results["B: GARCH(1,1)"]    = metrics_b
        strat_rets["B: GARCH(1,1)"] = metrics_b.pop("_strat_ret")
        n_trades_b = metrics_b.pop("_n_trades", 0)
        metrics_b.pop("_n_days", None)
        print(f"  Toplam Getiri: %{metrics_b['Toplam Getiri (%)']:.2f} | "
              f"Win Rate: %{metrics_b['Win Rate (%)']:.1f} | "
              f"İşlem: {n_trades_b}")
    except Exception as exc:
        print(f"  GARCH hatası: {exc}")
        results["B: GARCH(1,1)"] = {
            "Toplam Getiri (%)": 0.0, "Sharpe Oranı": 0.0,
            "Max Drawdown (%)": 0.0,  "Win Rate (%)": 0.0,
        }
        strat_rets["B: GARCH(1,1)"] = pd.Series(0, index=df_f.index[train_end:])

    # ── Benchmark: Buy & Hold ────────────────────────────────────────────────
    bh_ret   = df_f["log_ret"].iloc[train_end:]
    bh_total = (np.expm1(bh_ret.sum())) * 100
    strat_rets["Buy & Hold"] = bh_ret

    # ── Kıyaslama Tablosu ────────────────────────────────────────────────────
    print("\n[5/5] Karşılaştırma tablosu oluşturuluyor...")
    ordered = ["A: Klasik TA", "B: GARCH(1,1)", "C: XGBoost"]
    df_res  = pd.DataFrame({k: results[k] for k in ordered if k in results}).T
    df_res.index.name = "Strateji"

    # Tablo görünümü için format
    display_cols = ["Toplam Getiri (%)", "Sharpe Oranı", "Max Drawdown (%)", "Win Rate (%)"]
    df_display   = df_res[display_cols].copy()

    print("\n" + SEP)
    print(f"  PERFORMANS KARŞILAŞTIRMASI — Test Dönemi ({n-t} gün)")
    print(f"  Benchmark (Buy & Hold): %{bh_total:.2f}")
    print(SEP)
    print(df_display.to_string())
    print()

    # ── Şampiyon İlanı ───────────────────────────────────────────────────────
    # Çok kriterli skor (ağırlıklı sıra)
    df_score = df_display.copy()
    df_score["_score"] = (
        df_score["Toplam Getiri (%)"].rank() * 0.45 +
        df_score["Win Rate (%)"].rank()       * 0.30 +
        df_score["Sharpe Oranı"].rank()       * 0.25
    )
    champion    = df_score["_score"].idxmax()
    champ_data  = df_display.loc[champion]

    print(SEP)
    print(f"\n  *** ŞAMPİYON STRATEJİ: {champion} ***")
    print()
    print(f"  Toplam Getiri  : %{champ_data['Toplam Getiri (%)']:.2f}  "
          f"(Benchmark: %{bh_total:.2f})")
    print(f"  Sharpe Oranı   :  {champ_data['Sharpe Oranı']:.3f}")
    print(f"  Max Drawdown   : %{champ_data['Max Drawdown (%)']:.2f}")
    print(f"  Win Rate       : %{champ_data['Win Rate (%)']:.1f}")
    print("\n" + SEP)

    return df_res[display_cols], champion, strat_rets, df_f.index[train_end]

## Adım 5 — Görselleştirme

In [ ]:
def plot_results(strat_rets: dict, test_start_date, df_comparison: pd.DataFrame,
               champion: str):
    """Kümülatif getiri eğrileri ve performans çubuğu grafikleri."""

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = {
        "A: Klasik TA"   : "#2196F3",
        "B: GARCH(1,1)"  : "#FF9800",
        "C: XGBoost"     : "#4CAF50",
        "Buy & Hold"     : "#9E9E9E",
    }
    linestyles = {
        "A: Klasik TA"   : "-",
        "B: GARCH(1,1)"  : "--",
        "C: XGBoost"     : "-.",
        "Buy & Hold"     : ":",
    }

    # ── Sol: Kümülatif Getiri ─────────────────────────────────────────────────
    ax1 = axes[0]
    for name, ret_series in strat_rets.items():
        cum = cumulative_returns(ret_series)
        label_str = f"{name} ({'★ ' if name == champion else ''}%{(cum.iloc[-1]-1)*100:.1f})"
        ax1.plot(
            cum.index, cum.values,
            label=label_str,
            color=colors.get(name, "black"),
            linestyle=linestyles.get(name, "-"),
            linewidth=2.2 if name == champion else 1.5,
        )

    ax1.set_title("Kümülatif Getiri — Test Dönemi", fontsize=13, fontweight="bold")
    ax1.set_ylabel("Portföy Değeri (Başlangıç = 1)")
    ax1.axhline(1.0, color="black", lw=0.8, ls="--", alpha=0.4)
    ax1.legend(fontsize=9)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.2f}"))

    # ── Sağ: Metrik Karşılaştırma ─────────────────────────────────────────────
    ax2   = axes[1]
    metr  = ["Toplam Getiri (%)", "Sharpe Oranı", "Win Rate (%)"]
    strats = df_comparison.index.tolist()
    x      = np.arange(len(metr))
    width  = 0.22
    offsets= np.linspace(-(len(strats)-1)*width/2, (len(strats)-1)*width/2, len(strats))

    for i, strat in enumerate(strats):
        vals = [df_comparison.loc[strat, m] for m in metr]
        bars = ax2.bar(
            x + offsets[i], vals, width,
            label=strat,
            color=colors.get(strat, f"C{i}"),
            alpha=0.85,
            edgecolor="white",
        )
        for bar, val in zip(bars, vals):
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.2,
                f"{val:.1f}", ha="center", va="bottom", fontsize=7.5
            )

    ax2.set_title("Performans Metrikleri Karşılaştırması", fontsize=13, fontweight="bold")
    ax2.set_xticks(x)
    ax2.set_xticklabels(metr, fontsize=10)
    ax2.legend(fontsize=9)
    ax2.axhline(0, color="black", lw=0.8)

    plt.suptitle(
        f"Test Dönemi Başlangıcı: {str(test_start_date)[:10]}  |  "
        f"Şampiyon: {champion}",
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    plt.savefig("/tmp/strateji_karsilastirma.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Grafik kaydedildi: /tmp/strateji_karsilastirma.png")

## Demo — Gerçek Veri Yükleme veya Sentetik Test

Eğer elimizde gerçek bir `df` (temizlenmiş BIST verisi) varsa sadece son hücreyi çalıştırın.
Demo amaçlı sentetik veri üretmek için aşağıdaki hücreyi kullanın.

In [ ]:
# ── Seçenek 1: Gerçekçi Sentetik Veri Üret (Demo) ───────────────────────────
def create_synthetic_bist(
    n_days: int = 1000,
    start_price: float = 50.0,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Gerçek BIST hissesine benzer özellikler taşıyan sentetik OHLCV verisi üretir.
    Geometric Brownian Motion + rejim değişimi + hacim korelasyonu.
    """
    rng = np.random.default_rng(seed)

    # GBM parametreleri (günlük)
    mu    = 0.0005    # Hafif yukarı yönlü drift
    sigma = 0.018     # Günlük volatilite ~%1.8

    # Fiyat serisi
    log_rets  = mu + sigma * rng.standard_normal(n_days)
    close     = start_price * np.exp(np.cumsum(log_rets))

    # OHLC
    intraday_vol = sigma * 0.6
    high = close * np.exp( abs(rng.normal(0, intraday_vol, n_days)))
    low  = close * np.exp(-abs(rng.normal(0, intraday_vol, n_days)))
    open_ = close * np.exp(rng.normal(0, intraday_vol * 0.3, n_days))

    # Hacim (fiyat hareketleriyle zayıf korelasyon)
    vol_base = 1_000_000
    volume   = (vol_base * (1 + 0.5 * abs(log_rets) / sigma) *
                rng.lognormal(0, 0.3, n_days)).astype(int)

    # Endeks (XU100 benzeri): hisseyle %70 korelasyonlu
    index_log_rets = 0.7 * log_rets + 0.3 * (0.0003 + 0.012 * rng.standard_normal(n_days))
    endeks_close   = 9000 * np.exp(np.cumsum(index_log_rets))

    dates = pd.bdate_range("2021-01-04", periods=n_days, freq="B")

    df_syn = pd.DataFrame({
        "Open"        : open_,
        "High"        : high,
        "Low"         : low,
        "Close"       : close,
        "Volume"      : volume,
        "Endeks_Close": endeks_close,
    }, index=dates)
    df_syn.index.name = "Date"

    print(f"Sentetik veri oluşturuldu: {n_days} gün "
          f"({dates[0].date()} → {dates[-1].date()})")
    print(f"  Close: {close[0]:.2f} → {close[-1]:.2f} TL  "
          f"(kümülatif: %{(close[-1]/close[0]-1)*100:.1f})")
    return df_syn


# Demo için sentetik veri üret
df = create_synthetic_bist(n_days=1200, seed=42)
df.tail(3)

In [ ]:
# ── Seçenek 2: yfinance ile gerçek BIST hissesi yükle ───────────────────────
# Çalıştırmak için HISSE ve ENDEKS değişkenlerini düzenleyin.
# Bu hücreyi çalıştırmak opsiyoneldir — Demo hücresindeki df üzerine yazar.

RUN_YFINANCE = False   # True yaparak gerçek veri çekebilirsiniz

if RUN_YFINANCE:
    import yfinance as yf

    HISSE  = "THYAO.IS"
    ENDEKS = "^XU100"
    PERIOD = "5y"

    print(f"{HISSE} ve {ENDEKS} verisi çekiliyor...")
    hisse_df = yf.download(HISSE,  period=PERIOD, auto_adjust=True, progress=False)
    endeks_df = yf.download(ENDEKS, period=PERIOD, auto_adjust=True, progress=False)

    # MultiIndex düzelt
    if isinstance(hisse_df.columns, pd.MultiIndex):
        hisse_df.columns = hisse_df.columns.get_level_values(0)
    if isinstance(endeks_df.columns, pd.MultiIndex):
        endeks_df.columns = endeks_df.columns.get_level_values(0)

    df = hisse_df[["Open", "High", "Low", "Close", "Volume"]].copy()
    df["Endeks_Close"] = endeks_df["Close"]
    df.dropna(inplace=True)
    df.index.name = "Date"

    print(f"Yüklendi: {len(df)} gün ({df.index[0].date()} → {df.index[-1].date()})")
    print(f"  Son kapanış: {df['Close'].iloc[-1]:.2f} TL")

## Çalıştır — Tüm Sistemi Başlat

In [ ]:
# ── Tüm sistemi çalıştır ────────────────────────────────────────────────────
# df: temizlenmiş DataFrame (yukarıdaki demo veya kendi veriniz)

df_comparison, champion, strat_rets, test_start = run_comparison(df)

# Grafik
plot_results(strat_rets, test_start, df_comparison, champion)